Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Spark Structured Streaming metrics to OCI Monitoring

Spark's streaming gauges (`inputRate-total`, `processingRate-total`, `latency`, `eventTime-watermark`,
`states-rowsTotal`, `states-usedBytes`) exist only in Spark's own metrics system, and only when
`spark.sql.streaming.metricsEnabled=true`. This listener does **not** need that setting: it reads the
same values from the `StreamingQueryProgress` event of every micro-batch.
The cluster metrics AIDP publishes to your tenancy's OCI Monitoring (namespace `oracle_aidataplatform`)
cover tasks, jobs, shuffle, JVM and host resources, but **not** these streaming metrics.

This notebook attaches a `StreamingQueryListener` (`oci_streaming_listener.py` in this folder) that
posts the same numbers from every micro-batch to a **custom OCI Monitoring namespace** you own. It
then runs a short demo stream and reads the metrics back from OCI Monitoring.

**Prerequisites**
- An AIDP Credential Store credential of type *Secret Token* with keys `user`, `tenancy`,
  `fingerprint`, `region`, `key_content` (the PEM private key of an OCI API key). See the README.
- That user must be allowed to post metrics in the target compartment, e.g.
  `allow group <group> to use metrics in compartment <compartment> where target.metrics.namespace = 'custom_spark_streaming'`.
  To read the metrics back, it also needs `read metrics`.
- Spark 3.4+ (Python listeners). AIDP clusters run 3.5.0.

| OCI metric | Spark gauge / progress field |
|---|---|
| `inputRowsPerSecond` | `inputRate-total` |
| `processedRowsPerSecond` | `processingRate-total` |
| `triggerExecutionMs` | `latency` (`durationMs.triggerExecution`) |
| `watermarkEpochMs`, `watermarkLagMs` | `eventTime-watermark` (lag = trigger time − watermark) |
| `stateRowsTotal` | `states-rowsTotal` |
| `stateMemoryUsedBytes` | `states-usedBytes` |
| `numInputRows`, `batchDurationMs` | `numInputRows`, `durationMs.addBatch` |

Dimensions: `queryName`, `queryId`, `sparkAppId` (plus anything you add).

In [ ]:
# ---- configuration ----------------------------------------------------------------------
CREDENTIAL_NAME = "streaming_metrics_oci_api_key"   # Secret Token credential in the AIDP Credential Store
COMPARTMENT_ID = "<compartment ocid>"               # where the custom metrics are posted
NAMESPACE = "custom_spark_streaming"                # must not start with oci_ / oracle_
LISTENER_DIR = "/Workspace/streaming-metrics-to-oci-monitoring"   # folder holding oci_streaming_listener.py
RUN_DEMO_MINUTES = 4                                # 0 = attach the listener only, no demo stream
assert COMPARTMENT_ID.startswith(("ocid1.compartment.", "ocid1.tenancy.")), "set COMPARTMENT_ID"

In [ ]:
# ---- attach the listener (driver side) --------------------------------------------------------
import sys, importlib
if LISTENER_DIR not in sys.path:
    sys.path.insert(0, LISTENER_DIR)
import oci_streaming_listener as osl
importlib.reload(osl)

cfg = osl.oci_config_from_secrets(aidputils.secrets, CREDENTIAL_NAME)   # values are never printed
listener = osl.OciStreamingMetricsListener.from_config(
    cfg, COMPARTMENT_ID, namespace=NAMESPACE,
    dimensions={"sparkAppId": spark.sparkContext.applicationId})
spark.streams.addListener(listener)
print("listener attached; posting to", NAMESPACE, "every", listener.flush_seconds, "s")

## Demo stream

A `rate` source with a 10-second windowed count and a 30-second watermark, so state and watermark
metrics exist. It writes to a `noop` sink. The checkpoint must be an explicit `file:///` path: a bare
`/tmp/...` fails on AIDP with *"Wrong FS: compute:/tmp…"*.

In [ ]:
import shutil, time, uuid
from pyspark.sql import functions as F

QUERY_NAME = None
if RUN_DEMO_MINUTES > 0:
    QUERY_NAME = f"oci_metrics_demo_{uuid.uuid4().hex[:6]}"
    # a tiny stateful demo: 200 default shuffle partitions = 200 state stores -> ~8 s triggers
    spark.conf.set("spark.sql.shuffle.partitions", "4")
    src = spark.readStream.format("rate").option("rowsPerSecond", 200).load()
    agg = (src.withWatermark("timestamp", "30 seconds")
              .groupBy(F.window("timestamp", "10 seconds"), (F.col("value") % 10).alias("k")).count())
    q = (agg.writeStream.outputMode("update").format("noop").queryName(QUERY_NAME)
            .option("checkpointLocation", f"file:///tmp/{QUERY_NAME}_ckpt")
            .trigger(processingTime="5 seconds").start())
    t0 = time.time()
    while time.time() - t0 < RUN_DEMO_MINUTES * 60:
        time.sleep(30)
        print(time.strftime("%H:%M:%S"), "batch", (q.lastProgress or {}).get("batchId"), "listener", listener.stats)
    q.stop()
    shutil.rmtree(f"/tmp/{QUERY_NAME}_ckpt", ignore_errors=True)   # demo checkpoint on the driver
    print("stopped", QUERY_NAME)

In [ ]:
# ---- final flush (keep the listener attached instead if more streams follow in this session) ----
spark.streams.removeListener(listener)
print(listener.close())

## Read the metrics back from OCI Monitoring

In [ ]:
import datetime as dt, oci
mon = oci.monitoring.MonitoringClient(cfg)
end = dt.datetime.now(dt.timezone.utc); start = end - dt.timedelta(minutes=max(RUN_DEMO_MINUTES, 1) + 15)
flt = f'{{queryName = "{QUERY_NAME}"}}' if QUERY_NAME else ""
for name in ["inputRowsPerSecond", "processedRowsPerSecond", "triggerExecutionMs", "watermarkLagMs",
             "watermarkEpochMs", "stateRowsTotal", "stateMemoryUsedBytes", "numInputRows", "batchDurationMs"]:
    r = mon.summarize_metrics_data(COMPARTMENT_ID, oci.monitoring.models.SummarizeMetricsDataDetails(
        namespace=NAMESPACE, query=f"{name}[1m]{flt}.max()", start_time=start, end_time=end))
    pts = [p.value for s in r.data for p in s.aggregated_datapoints]
    print(f"{name:24} series={len(r.data)} points={len(pts)} " + (f"min={min(pts):.0f} max={max(pts):.0f} last={pts[-1]:.0f}" if pts else "-"))

## Using it in your own streaming job

```python
import sys; sys.path.insert(0, "/Workspace/streaming-metrics-to-oci-monitoring")
import oci_streaming_listener as osl
cfg = osl.oci_config_from_secrets(aidputils.secrets, "streaming_metrics_oci_api_key")
listener = osl.OciStreamingMetricsListener.from_config(
    cfg, "<compartment ocid>", dimensions={"pipeline": "orders"})
spark.streams.addListener(listener)    # before or after starting queries - every query is reported
# ... queries run ...
spark.streams.removeListener(listener); listener.close()
```

- **Posting is off the query path.** A background thread posts every `flush_seconds`, up to 50
  metric objects per call, so a slow or failing OCI call never delays a micro-batch.
- **Failures are counted, never raised:** see `listener.stats` (`posted`, `failed`, `dropped`, `last_error`).
- **Alarms:** in OCI Monitoring, use e.g. `watermarkLagMs[5m].max() > 600000` or
  `processedRowsPerSecond[5m].mean() < inputRowsPerSecond[5m].mean()`.
